# Student RAG Project - Guided Implementation

## Welcome!

In this project, you'll build a **RAG (Retrieval-Augmented Generation)** system that can answer questions about your documents.

### What You'll Learn:
- ✅ File I/O (reading documents)
- ✅ String manipulation (text chunking)
- ✅ Functions and parameters
- ✅ Lists and dictionaries
- ✅ Loops and conditionals
- ✅ Basic calculations and statistics

### What's Provided for You:
- ✅ Embedding model (converts text to numbers)
- ✅ Vector database (stores and searches embeddings)
- ✅ LLM connection (generates answers)

### Your Tasks:
You'll complete **TODO sections** marked with `# TODO:` comments.

Let's get started! 🚀

---
## Setup: Import Libraries

In [360]:
# Import the pre-built helper module
from rag_helpers import (
    EmbeddingModel,
    VectorDatabase,
    LLM,
    Timer,
    print_separator,
    print_search_results,
    print_rag_answer,
    check_setup
)

# Import standard Python libraries you'll use
from pathlib import Path
from typing import List, Dict
import json

# Check if everything is installed correctly
check_setup()

Checking setup...
✓ chromadb is installed
✓ sentence_transformers is installed
✓ requests is installed

✓ All required packages are installed!
You're ready to start!


True

In [ ]:
# vector_db.clear_database()

✓ Database cleared


---
## Configuration

Set up the basic settings for your RAG system.

In [ ]:
# TODO: Change this to point to YOUR documents folder
DOCS_FOLDER = r"my_docs\docs"
# Chunking settings (you can experiment with these!)
CHUNK_SIZE = 500     # How many characters per chunk
OVERLAP =  110        # How many characters overlap between chunks

# How many results to retrieve for each query
TOP_K = 5

print(f"Configuration:")
print(f"  Documents folder: {DOCS_FOLDER}")
print(f"  Chunk size: {CHUNK_SIZE} characters")
print(f"  Overlap: {OVERLAP} characters")
print(f"  Top-K results: {TOP_K}")

Configuration:
  Documents folder: C:\Users\Ami Shah\IS640\IS640_rag_project\docs
  Chunk size: 500 characters
  Overlap: 110 characters
  Top-K results: 5


---
## TODO #1: Document Loading

**Your Task:** Write a function to load all text files from a folder.

**What to do:**
1. Loop through all `.txt` files in the folder
2. Read each file's content
3. Store the content and filename in a dictionary
4. Return a list of these dictionaries

**Python concepts:** File I/O, loops, dictionaries, lists

In [362]:
def load_documents(folder_path: str) -> List[Dict[str, str]]:
    """
    Load all text documents from a folder.

    Args:
        folder_path: Path to folder containing .txt files

    Returns:
        List of dictionaries, each containing:
        - 'content': the text content of the file
        - 'filename': the name of the file

    Example:
        [
            {'content': 'This is doc 1...', 'filename': 'doc1.txt'},
            {'content': 'This is doc 2...', 'filename': 'doc2.txt'}
        ]
    """

    # TODO: Implement this function!
    # HINTS:
    # 1. Create an empty list to store documents
    # 2. Use Path(folder_path).glob("*.txt") to find all .txt files
    # 3. For each file:
    #    - Open it with open(file_path, 'r', encoding='utf-8')
    #    - Read the content with .read()
    #    - Create a dictionary with 'content' and 'filename'
    #    - Append to your list
    # 4. Return the list

    documents = []  # Start with empty list

    # Your code here:
    
    folder = Path(folder_path)

    for file_path in folder.glob("*.txt"):
        # Open and read the file
        # Create a dictionary
        # Add to documents list
        with open(file_path, 'r' , encoding = 'utf-8') as file:
            content = file.read()
            doc = {
                'content' : content,
                'filename' : file_path.name
            }
            documents.append(doc)          
        

    print(f"✓ Loaded {len(documents)} documents")
    return documents


# Test your function
documents = load_documents(DOCS_FOLDER)

# Display first document (if any were loaded)
if documents:
    print(f"\nFirst document: {documents[0]['filename']}")
    print(f"Content preview: {documents[0]['content'][:200]}...")
else:
    print("⚠️  No documents loaded! Check your folder path.")

✓ Loaded 7 documents

First document: 1_Fiber.txt
Content preview: Fiber
Fiber is a type of carbohydrate that the body can’t digest. Though most carbohydrates are broken down into sugar molecules called glucose, fiber cannot be broken down into sugar molecules, and i...


---
## TODO #2: Text Chunking Function

**Your Task:** Write a function to split long text into smaller chunks with overlap.

**Why?** Long documents are too big for embeddings. We need to split them into smaller pieces.

**What to do:**
1. Start at the beginning of the text
2. Take a chunk of `chunk_size` characters
3. Move forward by `chunk_size - overlap` characters
4. Repeat until you reach the end

**Python concepts:** String slicing, loops, lists

In [363]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> List[str]:
    """
    Split text into overlapping chunks.

    Args:
        text: The text to split
        chunk_size: Maximum characters per chunk
        overlap: How many characters to overlap between chunks

    Returns:
        List of text chunks

    Example:
        text = "This is a long document..."
        chunks = chunk_text(text, chunk_size=100, overlap=20)
        # Returns: ['This is a long...', 'long document...']
    """

    # TODO: Implement this function!
    # HINTS:
    # 1. Create an empty list to store chunks
    # 2. Start with position = 0
    # 3. While position < len(text):
    #    - Extract chunk from position to position+chunk_size
    #    - Add chunk to list (if not empty)
    #    - Move position forward by (chunk_size - overlap)
    # 4. Return the list of chunks

    chunks = []  # Start with empty list
    position = 0  # Start at beginning

    # Your code here:
    while position < len(text):
        # Extract a chunk
        # Add to chunks list
        # Move position forward
        End = position + chunk_size
        chunk = text[position:End]
        if chunk:
            chunks.append(chunk)
        position += chunk_size - overlap
    return chunks


# Test your chunking function
# test_text = "This is a test. " * 50  # Create a long test string
# test_chunks = chunk_text(test_text, chunk_size=100, overlap=20)

# print(f"Test text length: {len(test_text)} characters")
# print(f"Number of chunks: {len(test_chunks)}")
# print(f"\nFirst chunk: {test_chunks[0]}")
# if len(test_chunks) > 1:
#     print(f"Second chunk: {test_chunks[1]}")

---
## TODO #3: Process All Documents into Chunks

**Your Task:** Use your chunking function to split ALL documents into chunks and create metadata.

**What to do:**
1. Loop through each document
2. Chunk the document's content
3. For each chunk, create metadata (which file it came from, which chunk number)
4. Store everything in a list

**Python concepts:** Nested loops, dictionaries, enumerate

In [364]:
def process_documents(documents: List[Dict[str, str]],
                     chunk_size: int = 500,
                     overlap: int = 50) -> tuple:
    """
    Process all documents into chunks with metadata.

    Args:
        documents: List of document dictionaries
        chunk_size: Size of each chunk
        overlap: Overlap between chunks

    Returns:
        Tuple of (chunk_texts, chunk_metadatas)
        - chunk_texts: List of chunk strings
        - chunk_metadatas: List of metadata dictionaries
    """

    # TODO: Implement this function!
    # HINTS:
    # 1. Create two empty lists: chunk_texts and chunk_metadatas
    # 2. For each document:
    #    - Get the document's content and filename
    #    - Use your chunk_text() function to split it
    #    - For each chunk (use enumerate to get index):
    #      - Add chunk text to chunk_texts
    #      - Create metadata dict with 'source' and 'chunk_id'
    #      - Add metadata to chunk_metadatas
    # 3. Return both lists as a tuple

    chunk_texts = []
    chunk_metadatas = []

    # Your code here:
    for doc in documents:
        # Get document content and filename
        # Chunk the document
        # For each chunk:
        #   - Add chunk text to chunk_texts
        #   - Create metadata dictionary
        #   - Add metadata to chunk_metadatas
        content = doc['content']
        filename = doc['filename']
        chunks = chunk_text(content, chunk_size, overlap)
        for idx, chunk in enumerate(chunks):
            chunk_texts.append(chunk)
            metadata = {
                'source': filename,
                'chunk_id': idx
            }
            chunk_metadatas.append(metadata)

    print(f"✓ Created {len(chunk_texts)} chunks from {len(documents)} documents")
    return chunk_texts, chunk_metadatas


# Process all documents
chunk_texts, chunk_metadatas = process_documents(documents, CHUNK_SIZE, OVERLAP)

# Display example
if chunk_texts:
    print(f"\nExample chunk:")
    print(f"  Source: {chunk_metadatas[0]['source']}")
    print(f"  Chunk ID: {chunk_metadatas[0]['chunk_id']}")
    print(f"  Text: {chunk_texts[0][:200]}...")

✓ Created 24 chunks from 7 documents

Example chunk:
  Source: 1_Fiber.txt
  Chunk ID: 0
  Text: Fiber
Fiber is a type of carbohydrate that the body can’t digest. Though most carbohydrates are broken down into sugar molecules called glucose, fiber cannot be broken down into sugar molecules, and i...


---
## Pre-Built: Create Embeddings and Store in Database

This part uses the pre-built helpers. Just run these cells - no coding needed! ✨

In [365]:
# Initialize the embedding model (pre-built)
print("Initializing embedding model...")
embedder = EmbeddingModel()

# Create embeddings for all chunks (pre-built)
print("\nCreating embeddings...")
embeddings = embedder.embed_multiple(chunk_texts)
print(f"✓ Created {len(embeddings)} embeddings")

Initializing embedding model...
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
✓ Model loaded!

Creating embeddings...
Embedding 24 texts...
✓ Complete!
✓ Created 24 embeddings


In [366]:
# Initialize vector database (pre-built)

print("Initializing vector database...")
vector_db = VectorDatabase()
  # Clear any existing data


# Add chunks to database (pre-built)
print("\nAdding chunks to database...")
vector_db.add_chunks(chunk_texts, embeddings, chunk_metadatas)

Initializing vector database...
✓ Vector database initialized
  Collection: student_rag
  Current documents: 24

Adding chunks to database...
✓ Added 24 chunks to database


In [367]:
# Initialize LLM connection (pre-built)
print("Connecting to Ollama LLM...")
llm = LLM(model ="gemma3:1b-it-qat")

# Test the connection
print("\nTesting LLM connection...")
if llm.test_connection():
    print("✓ LLM is working!")
else:
    print("⚠️  LLM connection failed! Make sure Docker container is running.")

Connecting to Ollama LLM...
✓ LLM initialized: gemma3:1b-it-qat at http://127.0.0.1:11434

Testing LLM connection...
✓ LLM is working!


---
## TODO #4: RAG Query Function

**Your Task:** Write the main RAG function that ties everything together!

**What to do:**
1. Embed the user's question
2. Search the database for similar chunks
3. Build a prompt with the retrieved context
4. Ask the LLM to answer based on the context
5. Return the answer and metadata

**Python concepts:** Functions, string formatting, dictionaries

In [368]:
def rag_query(question: str, top_k: int = 3) -> Dict:
    """
    Answer a question using RAG (Retrieval-Augmented Generation).

    Args:
        question: The user's question
        top_k: How many chunks to retrieve

    Returns:
        Dictionary with:
        - 'question': the original question
        - 'answer': the LLM's answer
        - 'sources': list of source filenames
        - 'contexts': list of retrieved chunks
        - 'time': how long it took
    """

    # Start timer
    timer = Timer()
    timer.start()

    # TODO: Implement the RAG pipeline!
    # HINTS:
    # 1. Embed the question using: embedder.embed_text(question)
    # 2. Search database using: vector_db.search(query_embedding, top_k)
    # 3. Extract retrieved chunks and metadata from search results:
    #    - retrieved_chunks = results['documents'][0]
    #    - retrieved_metadata = results['metadatas'][0]
    # 4. Build context by joining chunks with newlines
    # 5. Create prompt (template below)
    # 6. Generate answer using: llm.generate_answer(prompt)
    # 7. Extract source filenames from metadata
    
    # 8. Return everything in a dictionary

    # Step 1: Embed question
    # Your code here
    query_embedding = embedder.embed_text(question)

 
    # Step 2: Search database
    # Your code here
    results = vector_db.search(query_embedding,top_k)


    # Step 3: Extract results
    # Your code here
    retrieved_chunks = results['documents'][0]  # Your code here
    retrieved_metadata = results['metadatas'][0]  # Your code here
    

    # Step 4: Build context
    # Your code here (join chunks with '\n\n')
    context = '\n\n'.join(results['documents'][0])

    # Step 5: Create prompt (use this template)
    prompt = f"""Answer the question based on the context below.

Context:{context}

Question: {question}

Answer:"""

    # Step 6: Generate answer
    answer = llm.generate_answer(prompt)  # Your code here

    # Step 7: Extract sources
    sources = [meta['source'] for meta in retrieved_metadata]  # Your code here (get 'source' from each metadata dict)

    # Stop timer
    elapsed_time = timer.stop()

    # Step 8: Return results
    return {
        'question': question,
        'answer': answer,
        'sources': sources,
        'contexts': retrieved_chunks,
        'time': elapsed_time
    }


print("✓ RAG query function defined!")

✓ RAG query function defined!


---
## Test Your RAG System!

Let's try asking some questions!

# FACTUAL QUESTION

In [369]:
# Test question 1
result = rag_query("How much protein do I need in a day?")

# print the result
print_rag_answer(
    result['question'],
    result['answer'],
    result['sources'],
    result['time']
)

======================== RAG ANSWER ========================

QUESTION: How much protein do I need in a day?

ANSWER:
The National Academy of Medicine recommends a minimum of 0.8 grams of protein for every kilogram of body weight per day, or just over 7 grams for every 20 pounds of body weight. For a 140-pound person, that means about 50 grams of protein each day. For a 200-pound person, that means about 70 grams of protein each day.

SOURCES: 1_Fiber.txt, 2_Protien.txt
TIME: 11.84 seconds


In [350]:
# Test question 2 
result = rag_query("How much Fiber do I need per day?")

# # Pretty print the result
print_rag_answer(
    result['question'],
    result['answer'],
    result['sources'],
    result['time']
)

======================== RAG ANSWER ========================

QUESTION: How much Fiber do I need per day?

ANSWER:
At least 25 to 35 grams of fiber per day for good health.

SOURCES: 1_Fiber.txt
TIME: 8.89 seconds


In [351]:
# Test question 3 
result = rag_query("What are examples of aerobic (cardiovascular) physical activities?")

# # Pretty print the result
print_rag_answer(
    result['question'],
    result['answer'],
    result['sources'],
    result['time']
)

======================== RAG ANSWER ========================

QUESTION: What are examples of aerobic (cardiovascular) physical activities?

ANSWER:
Walking, jogging, dancing, bicycling, basketball, soccer, swimming.

SOURCES: 3_Staying_Active.txt
TIME: 8.15 seconds


# INFERENCE QUESTION

In [370]:
# Test question 5 
result = rag_query("I'm worried about my heart rate and nervousness."
                    "At what daily dosage of my morning beverage does this anxiety risk increase,"
                    "and what breathing habit can I use to calm my mind immediately?")

# print the result
print_rag_answer(
    result['question'],
    result['answer'],
    result['sources'],
    result['time']
)

======================== RAG ANSWER ========================

QUESTION: I'm worried about my heart rate and nervousness.At what daily dosage of my morning beverage does this anxiety risk increase,and what breathing habit can I use to calm my mind immediately?

ANSWER:
According to the text, caffeine can increase anxiety at doses of 400 mg or more a day (about 4 cups of brewed coffee). 

To calm your mind immediately, you can use slow, deep breaths.

SOURCES: 4_Stress_and_Health.txt, 7_Caffeine.txt
TIME: 7.84 seconds


In [339]:
# Test question 4 
result = rag_query("How could a low-fiber diet contribute to increased eating during stress ?")

# # Pretty print the result
print_rag_answer(
    result['question'],
    result['answer'],
    result['sources'],
    result['time']
)

======================== RAG ANSWER ========================

QUESTION: How could a low-fiber diet contribute to increased eating during stress ?

ANSWER:
A low-fiber diet could contribute to increased eating during stress by leading to rapid, less mindful eating. Without sufficient fiber, the body doesn’t have the building blocks needed to properly regulate hunger and satiety signals, making it easier to overeat when stressed.

SOURCES: 1_Fiber.txt, 4_Stress_and_Health.txt
TIME: 4.31 seconds


# Edge Case

In [371]:
# Test question 6 
result = rag_query("What is the biochemical pathway linking exercise to reduced stress hormone levels?")

# # Pretty print the result
print_rag_answer(
    result['question'],
    result['answer'],
    result['sources'],
    result['time']
)

======================== RAG ANSWER ========================

QUESTION: What is the biochemical pathway linking exercise to reduced stress hormone levels?

ANSWER:
The provided text doesn't directly answer the question about the biochemical pathway linking exercise to reduced stress hormone levels. It focuses on the broader effects of exercise on health, stress, and digestion.

SOURCES: 4_Stress_and_Health.txt, 3_Staying_Active.txt
TIME: 8.83 seconds


# Funny - Edge Case

In [354]:

my_question =  "If I avoid screens before bed, does my phone get offended and refuse to wake me up tomorrow?"
# How many cups of coffee does it take before I legally become a jittery hummingbird?"  

result = rag_query(my_question)
print_rag_answer(
    result['question'],
    result['answer'],
    result['sources'],
    result['time']
)

======================== RAG ANSWER ========================

QUESTION: If I avoid screens before bed, does my phone get offended and refuse to wake me up tomorrow?

ANSWER:
No, your phone doesn’t get offended and refuse to wake you up tomorrow. The text states that stopping using electronic devices an hour before bed “closes off” the blue light emitted from screens, which can interfere with melatonin production.  Melatonin is a hormone that regulates sleep, and caffeine can disrupt this process.


SOURCES: 5_Sleep.txt, 7_Caffeine.txt
TIME: 19.56 seconds


---
## TODO #5: Create Test Dataset

**Your Task:** Create a list of test questions to evaluate your RAG system.

**What to do:**
1. Think of 10 questions your documents can answer
2. For each question, write the expected answer
3. Store them in a structured format

**Python concepts:** Lists, dictionaries, data structures

In [372]:
# TODO: Create your test dataset!
# HINTS:
# Create a list of dictionaries
# Each dictionary should have:
#   - 'question': the test question
#   - 'expected_answer': what you expect the answer to include
#   - 'category': type of question (factual, inferential, etc.)


test_questions = [
    # Example (replace with your own!):
    {
        'question': 'How much protein do I need in a day?',
        'expected_answer': 'Approximately 50 grams for a 140-pound person. For a 200-pound person, that’s about 70 grams.',
        'category': 'factual'
    },
 

    {
        'question': 'How much Fiber do I need per day?',
        'expected_answer': '25 to 35 grams of fiber per day.',
        'category': 'factual'
    },

    {
        'question': 'What are examples of aerobic (cardiovascular) physical activities?',
        'expected_answer': 'walking, jogging, dancing, bicycling, basketball, soccer, swimming',
        'category': 'factual'
    },

    {
        'question': 'I am worried about my heart rate and nervousness.At what daily dosage of my morning beverage does this anxiety risk increase,and what breathing habit can I use to calm my mind immediately?',
        'expected_answer': 'According to the context, a daily dosage of 400 mg or more of caffeine may increase anxiety and nervousness.  The text suggests breathing slowly, focusing on each inhale and exhale, as a calming technique.',
        'category': 'Inferential'
    },

    {
        'question': 'How could a low-fiber diet contribute to increased eating during stress ?',
        'expected_answer': 'Regular exercise. Physical activity will help to lower blood pressure and stress hormone levels. Aerobic exercise like walking and dancing increases breathing an.',
        'category': 'Inferential'
    },

    {

        'question': 'What is the biochemical pathway linking exercise to reduced stress hormone levels?',
        'expected_answer': "The context does not provide a specific biochemical pathway linking exercise to reduced stress hormone levels. It focuses on the general benefits of exercise, improved cardiovascular fitness, muscle-strengthening activity, and examples of aerobic exercise.",
        'category': 'Edge'
    },
 
    {
        
        'question': 'If I avoid screens before bed, does my phone get offended and refuse to wake me up tomorrow?',
        'expected_answer': "The context does not say your phone gets offended. It says that caffeine intake later in the day close to bedtime can interfere with good sleep quality.",
        'category': 'Edge'
    },
    {
        'question': 'How many cups of coffee does it take before I legally become a jittery hummingbird?',
        'expected_answer': "Based on the context, 4+ cups of coffee daily may cause symptoms like nervousness or a fast heart rate. Becoming a “jittery hummingbird” is a humorous exaggeration and not a real condition.",
        'category': 'Edge'
    },
    {
        'question': 'If fiber keeps me regular, can too much of it make me punctual for things I was not even planning to attend?',
        'expected_answer': "No. Fiber supports digestion but cannot make you punctual for events you did not plan to attend.",
        'category': 'Edge'
    },


    {

        'question': 'What is the simple change I can make to my grains (rice/bread) to boost my fiber intake?',
        'expected_answer': "You can boost your fiber intake from grains by replacing white rice, bread, and pasta with minimally processed brown rice and other whole grains like barley, millet, amaranth, farro, or others.",
        'category': 'Edge'
    }

    
    # Add 9 more questions here!
    # Your code here:

]



print(f"✓ Created {len(test_questions)} test questions")
print(f"\nExample question:")
print(f"  Q: {test_questions[0]['question']}")
print(f"  Expected: {test_questions[0]['expected_answer']}")

✓ Created 10 test questions

Example question:
  Q: How much protein do I need in a day?
  Expected: Approximately 50 grams for a 140-pound person. For a 200-pound person, that’s about 70 grams.


---
## TODO #6: Calculate Evaluation Metrics

**Your Task:** Write functions to measure how well your RAG system performs.

**Python concepts:** Functions, calculations, statistics

In [373]:
def calculate_average_latency(results: List[Dict]) -> float:
    """
    Calculate average response time.

    Args:
        results: List of result dictionaries (each has 'time' field)

    Returns:
        Average time in seconds
    """

    # TODO: Implement this function!
    # HINTS:
    # 1. Extract all 'time' values from results
    # 2. Sum them up
    # 3. Divide by the number of results
    # 4. Return the average

    # Your code here:

    if not results:
        return 0.0

    total_time = sum(r.get("time", 0) for r in results)
    avg_time = total_time / len(results)
    return avg_time

def count_successful_retrievals(results: List[Dict]) -> int:
    """
    Count how many queries successfully retrieved context.

    Args:
        results: List of result dictionaries

    Returns:
        Number of successful retrievals
    """

    # TODO: Implement this function!
    # HINTS:
    # 1. Start with count = 0
    # 2. For each result:
    #    - Check if 'contexts' is not empty
    #    - If yes, increment count
    # 3. Return count

    # Your code here:
    count = 0
    # Count successful retrievals
    
    for r in results:
        contexts = r.get("contexts", [])
        if contexts:  # Non-empty list means success
            count += 1
    return count
   


def get_all_sources(results: List[Dict]) -> List[str]:
    """
    Get unique list of all sources used.

    Args:
        results: List of result dictionaries

    Returns:
        List of unique source filenames
    """

    # TODO: Implement this function!
    # HINTS:
    # 1. Create an empty set (sets automatically keep unique values)
    # 2. For each result:
    #    - Get the 'sources' list
    #    - Add each source to the set
    # 3. Convert set to list and return

    # Your code here:
    all_sources = set()
    all_sources = set()
    for r in results:
        sources = r.get("sources", [])
        for s in sources:
            all_sources.add(s)

    return list(all_sources)
    # Collect all sources

    return list(all_sources)


print("✓ Evaluation functions defined!")

✓ Evaluation functions defined!


---
## TODO #7: Run Complete Evaluation

**Your Task:** Test your RAG system with all test questions and calculate metrics.

**Python concepts:** Loops, function calls, data aggregation

In [374]:
def run_evaluation(test_questions: List[Dict]) -> List[Dict]:
    """
    Run RAG system on all test questions.

    Args:
        test_questions: List of test question dictionaries

    Returns:
        List of result dictionaries
    """

    # TODO: Implement this function!
    # HINTS:
    # 1. Create empty list for results
    # 2. For each test question:
    #    - Get the question text
    #    - Call rag_query() with the question
    #    - Add result to results list
    # 3. Return results

    results = []

    # Your code here:
    for test in test_questions:
        # Get question
        question_text = test["question"]
        # Run RAG query
        result = rag_query(question_text)
        result["question"] = question_text
        # Store result
        results.append(result)  # Remove this 'pass' and write your code

    return results


# Run evaluation on all test questions
print("Running evaluation on all test questions...\n")
all_results = run_evaluation(test_questions)

print(f"\n✓ Completed {len(all_results)} tests")

Running evaluation on all test questions...


✓ Completed 10 tests


---
## Display Results

Show the evaluation metrics and results.

In [375]:
# Calculate metrics using your functions
avg_latency = calculate_average_latency(all_results)
successful_retrievals = count_successful_retrievals(all_results)
all_sources_used = get_all_sources(all_results)
hit_rate = successful_retrievals / len(all_results) if all_results else 0

# Display metrics
print_separator("EVALUATION RESULTS")
print(f"\nTotal Questions Tested: {len(all_results)}")
print(f"Successful Retrievals: {successful_retrievals}")
print(f"Hit Rate: {hit_rate:.2%}")
print(f"Average Latency: {avg_latency:.2f} seconds")
print(f"\nSources Used: {', '.join(all_sources_used)}")
print_separator()

==================== EVALUATION RESULTS ====================

Total Questions Tested: 10
Successful Retrievals: 10
Hit Rate: 100.00%
Average Latency: 8.61 seconds

Sources Used: 5_Sleep.txt, 4_Stress_and_Health.txt, 3_Staying_Active.txt, 7_Caffeine.txt, 1_Fiber.txt, 2_Protien.txt


In [376]:
# Display individual results
print("\nIndividual Test Results:\n")

for i, result in enumerate(all_results, 1):
    print(f"[Test {i}]")
    print(f"Question: {result['question']}")
    print(f"Answer: {result['answer'][:200]}...")
    print(f"Sources: {', '.join(set(result['sources']))}")
    print(f"Time: {result['time']:.2f}s")
    print("-" * 60)
    print()


Individual Test Results:

[Test 1]
Question: How much protein do I need in a day?
Answer: Approximately 50 grams for a 140-pound person, or 70 grams for a 200-pound person....
Sources: 1_Fiber.txt, 2_Protien.txt
Time: 6.76s
------------------------------------------------------------

[Test 2]
Question: How much Fiber do I need per day?
Answer: At least 25 to 35 grams of fiber per day for good health, but most Americans get only about 15 grams a day....
Sources: 1_Fiber.txt
Time: 7.39s
------------------------------------------------------------

[Test 3]
Question: What are examples of aerobic (cardiovascular) physical activities?
Answer: Walking, jogging, dancing, bicycling, basketball, soccer, swimming...
Sources: 3_Staying_Active.txt
Time: 6.82s
------------------------------------------------------------

[Test 4]
Question: I am worried about my heart rate and nervousness.At what daily dosage of my morning beverage does this anxiety risk increase,and what breathing habit can I use

---
## Save Your Results

Save your test results to a JSON file for your report.

In [377]:
# Save results to JSON file
results_summary = {
    'metrics': {
        'total_questions': len(all_results),
        'successful_retrievals': successful_retrievals,
        'hit_rate': hit_rate,
        'average_latency': avg_latency
    },
    'results': all_results
}

with open('evaluation_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print("✓ Results saved to 'evaluation_results.json'")

✓ Results saved to 'evaluation_results.json'


---
## Congratulations! 🎉

You've successfully built a RAG system!

### What You Accomplished:
✅ Loaded documents from files  
✅ Chunked text with overlap  
✅ Created a RAG query pipeline  
✅ Built a test dataset  
✅ Calculated evaluation metrics  
✅ Generated a results report  

### Next Steps:
- Try different chunk sizes and overlaps
- Add more test questions
- Experiment with different values for `top_k`
- Analyze which questions work best
- Write up your findings in a report

### For Your Report:
1. Describe your document collection
2. Explain your chunking strategy
3. Present your evaluation metrics
4. Show examples of good and bad answers
5. Discuss what you learned

Great job! 🚀